# 🚀 CI/CD with Flask — Complete Beginner's Guide

---

## What You Will Learn

This notebook covers **CI/CD (Continuous Integration / Continuous Deployment)** concepts using **Flask**, a popular Python web framework.

By the end of this notebook, you will understand:
1. What CI/CD is and why developers use it
2. How to build a simple Flask REST API
3. How to write automated tests for your Flask app
4. What a CI/CD pipeline looks like (GitHub Actions YAML)
5. How to containerize with Docker
6. How the full pipeline fits together

---

## 📖 Background: What is CI/CD?

Imagine a team of 5 developers all writing code for the same app. Without CI/CD:
- Each developer works in isolation for weeks
- When they combine code → chaos, conflicts, broken features
- Deployments are scary, manual, and error-prone

**CI/CD solves this:**

| Term | Meaning | Analogy |
|------|---------|---------|
| **CI** (Continuous Integration) | Automatically build & test code on every push | Spell-check while you type, not after you submit |
| **CD** (Continuous Delivery) | Code is always ready to deploy; human approves final push | The order is packed & ready — human clicks "ship" |
| **CD** (Continuous Deployment) | Every passing test automatically goes to production | Order ships the moment it's packed |

---

## 🗺️ The CI/CD Pipeline (Step by Step)

```
 Developer                GitHub                  CI Server (GitHub Actions)
 ─────────                ──────                  ──────────────────────────
 writes code  ──push──>  repository  ──trigger──> 1. Checkout code
                                                  2. Install dependencies
                                                  3. Run linter (code style)
                                                  4. Run tests (pytest)
                                                     │
                                                     ├─ PASS ──> 5. Build Docker image
                                                     │           6. Push to registry
                                                     │           7. Deploy to server ✅
                                                     │
                                                     └─ FAIL ──> Notify developer ❌
                                                                 (nothing deployed)
```

---

## 🧰 What is Flask?

Flask is a **micro web framework** for Python. "Micro" means it is small and simple — it gives you just what you need and nothing more.

Think of Flask like a **LEGO baseplate** — it gives you the foundation, and you add the pieces you need.

```
Django (full-stack)  →  Like buying a fully furnished house
Flask (micro)        →  Like buying an empty house, furnish it yourself
```

Flask is **synchronous** (WSGI). Each request is handled one at a time by a worker. This is simple and works great for most APIs.

---

Let's start building!

## Step 1: Install Flask and Testing Libraries

In a real CI/CD pipeline, these would be installed from a `requirements.txt` file.
Here in Colab, we install them directly.

```
requirements.txt (what a real project would have)
────────────────────────────────────────────────
Flask==3.0.0       ← the web framework
pytest==7.4.0      ← test runner
gunicorn==21.2.0   ← production WSGI server
```

In [ ]:
# ============================================================
# STEP 1: Install required packages
# ============================================================
# In a CI/CD pipeline, this step is called the 'BUILD' stage.
# The pipeline runner (e.g. GitHub Actions) runs:
#     pip install -r requirements.txt
# If ANY package fails to install, the pipeline STOPS here.
# This catches dependency errors before they reach production.
# ============================================================

!pip install flask pytest --quiet

print("✅ Packages installed successfully!")
print("In a CI/CD pipeline, this is the BUILD stage.")

## Step 2: Build a Flask REST API

We'll create a simple **Task Manager API** with these endpoints:

| Method | Endpoint | What it does |
|--------|----------|--------------|
| GET | `/` | Welcome message (health check) |
| GET | `/health` | Returns app status (used by load balancers in CI/CD) |
| GET | `/tasks` | List all tasks |
| POST | `/tasks` | Create a new task |
| GET | `/tasks/<id>` | Get one task by ID |
| DELETE | `/tasks/<id>` | Delete a task |

### What is a REST API?
A REST API is how applications talk to each other over the internet using HTTP.
- **GET** = Read data (like reading a book)
- **POST** = Create data (like writing a new page)
- **DELETE** = Remove data (like tearing out a page)

In [ ]:
# ============================================================
# STEP 2: Write the Flask Application
# ============================================================
# We write the app to a file called app.py.
# In a real project, this file lives in your repository.
#
# We use the %%writefile magic to create the file in Colab.
# This is the same app.py that would be deployed in production.
# ============================================================

# The %%writefile magic saves the cell contents to app.py
# Everything below the first line is written to the file
app_code = '''
# ============================================================
# app.py — The Flask Application
# ============================================================
# Flask is imported from the flask package.
# jsonify converts Python dicts to JSON responses.
# request lets us read incoming request data.
# ============================================================

from flask import Flask, jsonify, request

# ─────────────────────────────────────────────────────────────
# CREATE THE FLASK APP
# ─────────────────────────────────────────────────────────────
# Flask(__name__) creates the application.
# __name__ tells Flask where to find files relative to this script.
# We store it in a variable called 'app' (this is the convention).
# ─────────────────────────────────────────────────────────────
app = Flask(__name__)

# ─────────────────────────────────────────────────────────────
# IN-MEMORY DATABASE (for demo purposes)
# ─────────────────────────────────────────────────────────────
# In production, you'd use a real database like PostgreSQL.
# Here we use a Python dictionary (key=id, value=task data).
# This resets every time the server restarts — that's fine for demos.
# ─────────────────────────────────────────────────────────────
tasks = {}        # Dictionary to store tasks: {id: task_data}
next_id = [1]     # We use a list so nested functions can modify it


# ─────────────────────────────────────────────────────────────
# ROUTE 1: Root / Welcome
# ─────────────────────────────────────────────────────────────
# @app.route('/') is a DECORATOR — it tells Flask:
#   "When someone visits http://yourserver/, run this function."
# The function must return a response (JSON, HTML, or plain text).
# ─────────────────────────────────────────────────────────────
@app.route('/')
def index():
    """Root endpoint — returns a welcome message."""
    return jsonify({
        "message": "Welcome to the Task Manager API",
        "version": "1.0.0",
        "framework": "Flask"
    })


# ─────────────────────────────────────────────────────────────
# ROUTE 2: Health Check
# ─────────────────────────────────────────────────────────────
# CRITICAL FOR CI/CD: Every production app needs a /health endpoint.
# Load balancers and orchestrators (like Kubernetes) call /health
# to check if the app is alive. If it returns anything other than
# 200 OK, the container is restarted or traffic is redirected.
# This endpoint MUST be fast and MUST NOT depend on a database.
# ─────────────────────────────────────────────────────────────
@app.route('/health')
def health():
    """Health check endpoint for CI/CD load balancers and monitoring."""
    return jsonify({
        "status": "healthy",
        "framework": "Flask"
    }), 200  # 200 = HTTP status code for OK


# ─────────────────────────────────────────────────────────────
# ROUTE 3: List all tasks (GET /tasks)
# ─────────────────────────────────────────────────────────────
# GET requests retrieve data — they should never change data.
# We convert the dictionary values to a list for the JSON response.
# ─────────────────────────────────────────────────────────────
@app.route('/tasks', methods=['GET'])
def get_tasks():
    """Return a list of all tasks."""
    task_list = list(tasks.values())  # dict.values() → list of task dicts
    return jsonify({
        "tasks": task_list,
        "total": len(task_list)  # How many tasks exist
    })


# ─────────────────────────────────────────────────────────────
# ROUTE 4: Create a task (POST /tasks)
# ─────────────────────────────────────────────────────────────
# POST requests send data TO the server to create something new.
# The client sends JSON in the request body, e.g.:
#   {"title": "Buy groceries", "done": false}
#
# request.get_json() reads the JSON body sent by the client.
# We validate that the required 'title' field is present.
# HTTP 400 = Bad Request (client sent bad data)
# HTTP 201 = Created (resource was successfully created)
# ─────────────────────────────────────────────────────────────
@app.route('/tasks', methods=['POST'])
def create_task():
    """Create a new task. Expects JSON body with 'title' field."""
    data = request.get_json()  # Parse incoming JSON body

    # --- Input Validation ---
    # Always validate user input before saving it!
    # If 'title' is missing or empty, return a 400 error.
    if not data or 'title' not in data or not data['title'].strip():
        return jsonify({"error": "'title' field is required and cannot be empty"}), 400

    # --- Create the task ---
    task_id = next_id[0]  # Get the current ID number
    next_id[0] += 1       # Increment for the next task

    task = {
        "id": task_id,
        "title": data['title'].strip(),          # .strip() removes leading/trailing spaces
        "done": data.get('done', False),          # .get() returns False if 'done' not provided
        "priority": data.get('priority', 'low')  # Default priority is 'low'
    }

    tasks[task_id] = task  # Save to our in-memory dictionary

    return jsonify({"task": task, "message": "Task created"}), 201  # 201 = Created


# ─────────────────────────────────────────────────────────────
# ROUTE 5: Get one task (GET /tasks/<id>)
# ─────────────────────────────────────────────────────────────
# <int:task_id> is a URL parameter — Flask extracts the integer
# from the URL and passes it to the function automatically.
# Example: GET /tasks/3 → task_id = 3
#
# HTTP 404 = Not Found (the requested resource doesn't exist)
# ─────────────────────────────────────────────────────────────
@app.route('/tasks/<int:task_id>', methods=['GET'])
def get_task(task_id):
    """Return a single task by its ID."""
    task = tasks.get(task_id)  # Look up the task in our dictionary

    if task is None:  # Task not found
        return jsonify({"error": f"Task with id {task_id} not found"}), 404

    return jsonify({"task": task})


# ─────────────────────────────────────────────────────────────
# ROUTE 6: Delete a task (DELETE /tasks/<id>)
# ─────────────────────────────────────────────────────────────
# DELETE requests remove a resource.
# We use dict.pop() which removes and returns the item.
# If the ID doesn't exist, pop() returns None (the default).
# ─────────────────────────────────────────────────────────────
@app.route('/tasks/<int:task_id>', methods=['DELETE'])
def delete_task(task_id):
    """Delete a task by its ID."""
    removed = tasks.pop(task_id, None)  # Remove from dict; None if not found

    if removed is None:
        return jsonify({"error": f"Task with id {task_id} not found"}), 404

    return jsonify({"message": f"Task {task_id} deleted", "task": removed})


# ─────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────
# This block runs ONLY when you execute this file directly:
#   python app.py
# It does NOT run when Flask is started by gunicorn or pytest.
#
# debug=True means:
#   - Auto-reloads when you change the code (great for development)
#   - Shows detailed error pages (NEVER use in production!)
# ─────────────────────────────────────────────────────────────
if __name__ == '__main__':
    app.run(debug=True, port=5000)
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print("✅ app.py created!")
print("This is the Flask application with 6 API endpoints.")

## Step 3: Test the Flask App Manually

Before writing automated tests, let's manually call the API to understand what it returns.

In CI/CD, we eventually automate all these checks — but understanding what a passing request looks like helps you write good tests.

In [ ]:
# ============================================================
# STEP 3: Manual Testing — Call the API Directly
# ============================================================
# Flask provides a 'test client' that lets you call your API
# WITHOUT starting a real server. This is perfect for testing!
#
# app.test_client() creates a fake browser/HTTP client.
# client.get('/route') makes a GET request to that route.
# client.post('/route', json={...}) makes a POST request with data.
# ============================================================

import sys, importlib

# Import our Flask app from app.py
# We need to reload it if it was imported before
if 'app' in sys.modules:
    del sys.modules['app']

import app as flask_app_module

# Get the Flask app object and reset state for clean demo
flask_app = flask_app_module.app
flask_app_module.tasks.clear()
flask_app_module.next_id[0] = 1

# Create the test client
# Think of 'client' as a virtual browser that talks to our API
client = flask_app.test_client()

print("=" * 55)
print("MANUAL TEST 1: GET / (Welcome message)")
print("=" * 55)
response = client.get('/')
# response.status_code: the HTTP status (200=OK, 404=Not Found, etc.)
# response.get_json(): parse the JSON body into a Python dict
print(f"Status Code: {response.status_code}")
print(f"Response Body: {response.get_json()}")

print()
print("=" * 55)
print("MANUAL TEST 2: GET /health (Health check)")
print("=" * 55)
response = client.get('/health')
print(f"Status Code: {response.status_code}")
print(f"Response Body: {response.get_json()}")
print("   ↑ The CI/CD pipeline checks this is 200 before deploying")

print()
print("=" * 55)
print("MANUAL TEST 3: POST /tasks (Create a task)")
print("=" * 55)
response = client.post('/tasks', json={'title': 'Learn CI/CD', 'priority': 'high'})
print(f"Status Code: {response.status_code}")
print(f"Response Body: {response.get_json()}")
print("   ↑ 201 = Created (resource was successfully made)")

print()
print("=" * 55)
print("MANUAL TEST 4: POST /tasks (Missing title — should fail)")
print("=" * 55)
response = client.post('/tasks', json={'priority': 'high'})  # Missing 'title'!
print(f"Status Code: {response.status_code}")
print(f"Response Body: {response.get_json()}")
print("   ↑ 400 = Bad Request (client sent invalid data)")

print()
print("=" * 55)
print("MANUAL TEST 5: GET /tasks (List all tasks)")
print("=" * 55)
response = client.get('/tasks')
print(f"Status Code: {response.status_code}")
print(f"Response Body: {response.get_json()}")

## Step 4: Write Automated Tests (The Heart of CI/CD)

### Why Automated Tests?

Manual testing (what we did above) is fine when you're learning. But imagine:
- Your app has 200 API endpoints
- You make a change to one function
- Did it break any of the other 199 endpoints?

You can't manually test 200 endpoints every time. **Automated tests do this in seconds.**

### The CI/CD Test Stage
```
Developer pushes code → GitHub → CI Pipeline runs:
    $ pytest tests/            ← runs ALL test files
    
    PASSED tests/test_app.py::test_health ✅
    PASSED tests/test_app.py::test_create_task ✅
    FAILED tests/test_app.py::test_get_task ❌  ← pipeline STOPS here
    
    → Developer gets notified. Nothing is deployed.
```

### pytest Basics
- **Test file**: any file starting with `test_`
- **Test function**: any function starting with `test_`
- **Assertion**: `assert <condition>` — if False, the test fails
- **Fixture**: a function that sets up shared test data (like our `client`)

In [ ]:
# ============================================================
# STEP 4: Write the Test File
# ============================================================
# pytest looks for files named test_*.py or *_test.py.
# Inside, it looks for functions named test_*.
# Each test function tests ONE specific behavior.
#
# The key principle: each test should be INDEPENDENT.
# Tests should not depend on each other's state.
# ============================================================

test_code = '''
# ============================================================
# test_app.py — Automated Tests for the Flask Task Manager API
# ============================================================
# This file is what the CI/CD pipeline runs with: pytest test_app.py
# Every function that starts with 'test_' is a test case.
# If any assertion fails, pytest marks that test as FAILED.
# ============================================================

import pytest
import app as flask_app_module

# ─────────────────────────────────────────────────────────────
# PYTEST FIXTURE: client
# ─────────────────────────────────────────────────────────────
# A fixture is a setup function that pytest calls automatically
# before each test that requests it.
#
# @pytest.fixture tells pytest: "This is a reusable setup."
#
# Any test function that has 'client' as a parameter will
# automatically get a fresh test client with an empty task list.
#
# This ensures tests are INDEPENDENT — one test\' state
# does not affect another test.
# ─────────────────────────────────────────────────────────────
@pytest.fixture
def client():
    """Create a fresh test client before each test."""
    flask_app = flask_app_module.app
    flask_app.config['TESTING'] = True  # Enable test mode (better error messages)

    # Reset the in-memory database before each test
    # This ensures each test starts with a clean slate
    flask_app_module.tasks.clear()
    flask_app_module.next_id[0] = 1

    # yield: the test runs here. After the test, cleanup code (if any) goes below.
    with flask_app.test_client() as test_client:
        yield test_client  # Give the test client to the test function


# ─────────────────────────────────────────────────────────────
# TEST 1: Root endpoint
# ─────────────────────────────────────────────────────────────
# Test naming convention: test_<what_you_are_testing>
# Each test checks ONE behavior.
# 'assert' checks that something is True. If not → test FAILS.
# ─────────────────────────────────────────────────────────────
def test_root_endpoint(client):
    """Test that the root endpoint returns a welcome message."""
    response = client.get('/')

    # Check the HTTP status code
    assert response.status_code == 200, "Root should return 200 OK"

    # Check the response body
    data = response.get_json()
    assert 'message' in data, "Response should have a message field"
    assert 'framework' in data, "Response should mention the framework"
    assert data['framework'] == 'Flask', "Framework should be Flask"


# ─────────────────────────────────────────────────────────────
# TEST 2: Health check endpoint
# ─────────────────────────────────────────────────────────────
# The /health endpoint is critical for CI/CD.
# If this test fails, the pipeline won\'t deploy.
# ─────────────────────────────────────────────────────────────
def test_health_check(client):
    """Test that the health endpoint returns healthy status."""
    response = client.get('/health')

    assert response.status_code == 200
    data = response.get_json()
    assert data['status'] == 'healthy', "Health check must return \'healthy\'"


# ─────────────────────────────────────────────────────────────
# TEST 3: Get tasks when empty
# ─────────────────────────────────────────────────────────────
def test_get_tasks_empty(client):
    """Test that getting tasks on an empty list returns an empty array."""
    response = client.get('/tasks')

    assert response.status_code == 200
    data = response.get_json()
    assert data['tasks'] == [], "Empty database should return empty list"
    assert data['total'] == 0, "Total should be 0 when no tasks exist"


# ─────────────────────────────────────────────────────────────
# TEST 4: Create a task successfully
# ─────────────────────────────────────────────────────────────
# We test the HAPPY PATH: what happens when everything goes right.
# ─────────────────────────────────────────────────────────────
def test_create_task_success(client):
    """Test creating a valid task returns 201 and the task data."""
    task_data = {'title': 'Learn pytest', 'priority': 'high'}

    response = client.post('/tasks', json=task_data)

    assert response.status_code == 201, "Creating a task should return 201 Created"

    data = response.get_json()
    assert 'task' in data, "Response should contain the created task"
    assert data['task']['title'] == 'Learn pytest', "Task title should match input"
    assert data['task']['priority'] == 'high', "Priority should be saved"
    assert data['task']['done'] == False, "New task should not be done"
    assert 'id' in data['task'], "Task should have an auto-assigned ID"


# ─────────────────────────────────────────────────────────────
# TEST 5: Create a task with missing title (ERROR PATH)
# ─────────────────────────────────────────────────────────────
# We test the UNHAPPY PATH: what happens when input is bad.
# Testing error paths is just as important as happy paths!
# ─────────────────────────────────────────────────────────────
def test_create_task_missing_title(client):
    """Test that creating a task without a title returns 400 Bad Request."""
    response = client.post('/tasks', json={'priority': 'high'})  # No title!

    assert response.status_code == 400, "Missing title should return 400 Bad Request"
    data = response.get_json()
    assert 'error' in data, "Error response should have an \'error\' field"


# ─────────────────────────────────────────────────────────────
# TEST 6: Create a task with empty title
# ─────────────────────────────────────────────────────────────
def test_create_task_empty_title(client):
    """Test that an empty title is rejected."""
    response = client.post('/tasks', json={'title': '   '})  # Just whitespace

    assert response.status_code == 400


# ─────────────────────────────────────────────────────────────
# TEST 7: Get a specific task
# ─────────────────────────────────────────────────────────────
# This test first creates a task, then retrieves it.
# It tests that the data survives being stored and retrieved.
# ─────────────────────────────────────────────────────────────
def test_get_specific_task(client):
    """Test retrieving a specific task by ID."""
    # First, create a task
    create_response = client.post('/tasks', json={'title': 'Deploy to production'})
    task_id = create_response.get_json()['task']['id']

    # Now retrieve it by ID
    get_response = client.get(f'/tasks/{task_id}')

    assert get_response.status_code == 200
    data = get_response.get_json()
    assert data['task']['id'] == task_id
    assert data['task']['title'] == 'Deploy to production'


# ─────────────────────────────────────────────────────────────
# TEST 8: Get a non-existent task
# ─────────────────────────────────────────────────────────────
def test_get_nonexistent_task(client):
    """Test that requesting a task that doesn\'t exist returns 404."""
    response = client.get('/tasks/9999')  # This ID doesn\'t exist

    assert response.status_code == 404, "Non-existent task should return 404 Not Found"
    data = response.get_json()
    assert 'error' in data


# ─────────────────────────────────────────────────────────────
# TEST 9: Delete a task
# ─────────────────────────────────────────────────────────────
def test_delete_task(client):
    """Test that deleting a task removes it from the list."""
    # Create a task first
    create_response = client.post('/tasks', json={'title': 'Task to delete'})
    task_id = create_response.get_json()['task']['id']

    # Delete it
    delete_response = client.delete(f'/tasks/{task_id}')
    assert delete_response.status_code == 200

    # Verify it\'s gone
    get_response = client.get(f'/tasks/{task_id}')
    assert get_response.status_code == 404, "Deleted task should return 404"


# ─────────────────────────────────────────────────────────────
# TEST 10: Multiple tasks workflow (Integration test)
# ─────────────────────────────────────────────────────────────
# Integration tests check that multiple parts work together.
# This simulates a real user creating and listing multiple tasks.
# ─────────────────────────────────────────────────────────────
def test_multiple_tasks_workflow(client):
    """Integration test: create multiple tasks and verify the list."""
    # Create three tasks
    titles = [\'Set up CI pipeline\', \'Write tests\', \'Deploy to production\']
    for title in titles:
        client.post(\'/tasks\', json={\'title\': title})

    # Check the list has all three
    response = client.get(\'/tasks\')
    data = response.get_json()

    assert data[\'total\'] == 3, "Should have 3 tasks"
    retrieved_titles = [t[\'title\'] for t in data[\'tasks\']]
    for title in titles:
        assert title in retrieved_titles, f\'\'{title}\' should be in the task list\'
'''

with open('test_app.py', 'w') as f:
    f.write(test_code)

print("✅ test_app.py created!")
print("This file contains 10 test cases covering happy paths and error paths.")

## Step 5: Run the Tests (Simulating the CI Pipeline)

This is the most important step in CI/CD. Every time a developer pushes code, the pipeline runs exactly this command:

```bash
pytest test_app.py -v
```

- `-v` = verbose (show each test name and result)
- A green `PASSED` means ✅
- A red `FAILED` means ❌ — deployment is blocked

In [ ]:
# ============================================================
# STEP 5: Run pytest — This IS the CI Pipeline Test Stage
# ============================================================
# In GitHub Actions, the YAML would contain:
#   - name: Run tests
#     run: pytest test_app.py -v
#
# If pytest exits with code 0 → all tests passed → pipeline continues
# If pytest exits with code 1 → some tests failed → pipeline STOPS
# ============================================================

print("🔄 Running the CI/CD test stage...")
print("Command: pytest test_app.py -v")
print("=" * 60)

import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest', 'test_app.py', '-v', '--tb=short'],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

print("=" * 60)
if result.returncode == 0:
    print("✅ ALL TESTS PASSED — Pipeline would proceed to deployment!")
else:
    print("❌ TESTS FAILED — Pipeline would STOP. Nothing gets deployed.")

## Step 6: The GitHub Actions CI/CD Pipeline (YAML)

In real projects, you create a `.github/workflows/ci.yml` file.
This YAML file defines the pipeline that GitHub runs automatically on every push.

Let's look at a complete pipeline for our Flask app:

In [ ]:
# ============================================================
# STEP 6: GitHub Actions YAML — The CI/CD Pipeline Definition
# ============================================================
# This YAML file is placed at:
#   .github/workflows/ci.yml
# in your GitHub repository.
#
# GitHub reads this file and runs it as a pipeline.
# ============================================================

github_actions_yaml = """
# ============================================================
# .github/workflows/ci.yml
# CI/CD Pipeline for Flask Task Manager API
# ============================================================

# The 'name' appears in GitHub's Actions tab
name: Flask CI/CD Pipeline

# ─────────────────────────────────────────────────────────────
# TRIGGERS: When does this pipeline run?
# ─────────────────────────────────────────────────────────────
on:
  push:
    branches: [ main, develop ]  # Run on pushes to main or develop
  pull_request:
    branches: [ main ]           # Run on pull requests targeting main

# ─────────────────────────────────────────────────────────────
# JOBS: Groups of steps that run on a runner (virtual machine)
# ─────────────────────────────────────────────────────────────
jobs:

  # ── JOB 1: Test ──────────────────────────────────────────
  # This job runs on every push. If it fails, job 2 doesn't run.
  test:
    name: Build and Test
    runs-on: ubuntu-latest  # GitHub spins up a fresh Ubuntu VM

    steps:
      # Step 1: Get your code onto the runner
      - name: Checkout code
        uses: actions/checkout@v4  # Built-in GitHub Action

      # Step 2: Set up Python
      - name: Set up Python 3.11
        uses: actions/setup-python@v4
        with:
          python-version: '3.11'

      # Step 3: BUILD stage — Install dependencies
      # This is like 'pip install -r requirements.txt'
      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt

      # Step 4: LINT stage — Check code style
      # flake8 checks for syntax errors and style violations
      # A linting failure means bad code won't be deployed
      - name: Lint with flake8
        run: |
          pip install flake8
          flake8 app.py --max-line-length=100

      # Step 5: TEST stage — Run all tests
      # If any test fails, the pipeline stops here
      - name: Run tests with pytest
        run: pytest test_app.py -v --tb=short

  # ── JOB 2: Deploy ────────────────────────────────────────
  # This job only runs if the 'test' job passed.
  # 'needs: test' creates a dependency between jobs.
  deploy:
    name: Deploy to Production
    runs-on: ubuntu-latest
    needs: test                 # ← Only runs if 'test' job succeeded
    if: github.ref == 'refs/heads/main'  # Only deploy from main branch

    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      # Build Docker image
      # The Dockerfile is explained in the next cell
      - name: Build Docker image
        run: docker build -t flask-task-api:latest .

      # Push to Docker Hub (registry)
      # DOCKER_TOKEN is a secret stored in GitHub Settings → Secrets
      - name: Push to Docker Hub
        run: |
          echo ${{ secrets.DOCKER_TOKEN }} | docker login -u myusername --password-stdin
          docker push myusername/flask-task-api:latest

      # Deploy to server (example: Heroku or Railway)
      - name: Deploy to Heroku
        run: |
          heroku container:push web --app my-flask-app
          heroku container:release web --app my-flask-app
"""

print(github_actions_yaml)
print("─" * 60)
print("KEY TAKEAWAYS:")
print("  1. 'on: push' triggers the pipeline on every code push")
print("  2. 'needs: test' creates job dependencies (test → deploy)")
print("  3. If any step fails (non-zero exit code), pipeline STOPS")
print("  4. Secrets (API keys) are stored in GitHub, not in code")
print("  5. Each job runs on a fresh virtual machine")

## Step 7: Dockerfile — Packaging the Flask App

In [ ]:
# ============================================================
# STEP 7: Dockerfile — Containerization
# ============================================================
# Docker packages your app into a container — a lightweight,
# self-contained environment that runs identically everywhere.
#
# Without Docker:
#   Developer: "It works on my machine!"
#   Server:    "It crashes here!"
#
# With Docker:
#   Same container runs everywhere. Problem solved.
#
# The Dockerfile is a recipe. 'docker build' follows the recipe.
# ============================================================

dockerfile = """
# ============================================================
# Dockerfile for Flask Task Manager API
# ============================================================

# FROM: Start with an official Python base image.
# 'python:3.11-slim' is Python 3.11 on minimal Debian Linux.
# 'slim' means fewer pre-installed packages → smaller image size.
FROM python:3.11-slim

# WORKDIR: Set the working directory inside the container.
# All following commands run from /app.
# If /app doesn't exist, Docker creates it.
WORKDIR /app

# COPY requirements.txt first (before the rest of the code).
# WHY? Docker caches layers. If requirements.txt hasn't changed,
# Docker skips the pip install step (much faster rebuilds).
COPY requirements.txt .

# RUN: Execute a command during the build process.
# --no-cache-dir: Don't store pip's download cache (saves space)
RUN pip install --no-cache-dir -r requirements.txt

# COPY the rest of your application code into the container
COPY . .

# EXPOSE: Document which port the app listens on.
# This is informational — it doesn't actually open the port.
# The actual port mapping is done in 'docker run -p 5000:5000'.
EXPOSE 5000

# CMD: The command to run when the container starts.
# gunicorn is a production WSGI server (NOT Flask's built-in server).
# Flask's built-in server (app.run) is for DEVELOPMENT ONLY.
#
# gunicorn flags explained:
#   --workers 4       → 4 parallel worker processes
#   --bind 0.0.0.0:5000  → listen on all interfaces, port 5000
#   app:app           → from file 'app.py', use object named 'app'
CMD ["gunicorn", "--workers", "4", "--bind", "0.0.0.0:5000", "app:app"]
"""

print(dockerfile)

# Also show requirements.txt
print("─" * 60)
print("requirements.txt (what goes in this file):")
print("""
Flask==3.0.0
gunicorn==21.2.0
pytest==7.4.0
""")
print("─" * 60)
print("Docker commands to build and run:")
print("  docker build -t flask-task-api .    ← build the image")
print("  docker run -p 5000:5000 flask-task-api  ← run the container")
print("  curl http://localhost:5000/health   ← test it!")

## Step 8: Final Summary — The Complete CI/CD Flow

In [ ]:
# ============================================================
# STEP 8: Summary and Key Takeaways
# ============================================================

print("""
╔══════════════════════════════════════════════════════════╗
║         CI/CD with Flask — Complete Summary              ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  WHAT WE BUILT:                                          ║
║  ─────────────                                           ║
║  ✅ Flask REST API with 6 endpoints                      ║
║  ✅ 10 automated pytest tests                            ║
║  ✅ GitHub Actions YAML pipeline                         ║
║  ✅ Production Dockerfile with gunicorn                  ║
║                                                          ║
║  THE CI/CD PIPELINE STAGES:                             ║
║  ──────────────────────────                              ║
║  1. CODE    → Developer pushes to GitHub                 ║
║  2. BUILD   → pip install -r requirements.txt            ║
║  3. LINT    → flake8 app.py                              ║
║  4. TEST    → pytest test_app.py -v                      ║
║  5. PACKAGE → docker build -t myapp .                    ║
║  6. DEPLOY  → Push image, release to server              ║
║                                                          ║
║  KEY FLASK CONCEPTS:                                     ║
║  ───────────────────                                     ║
║  • @app.route()    → Maps URL to function                ║
║  • request.get_json() → Read POST body                   ║
║  • jsonify()       → Return JSON response                ║
║  • HTTP codes: 200 OK, 201 Created, 400 Bad Request,     ║
║               404 Not Found                              ║
║  • test_client()   → Test without a running server       ║
║  • gunicorn        → Production WSGI server              ║
║                                                          ║
║  FLASK vs FASTAPI (Preview):                             ║
║  ──────────────────────────                              ║
║  Flask is synchronous (WSGI)                             ║
║  FastAPI is async (ASGI) — see the FastAPI notebook!     ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝
""")

print("Next steps:")
print("  1. Open the FastAPI notebook to see how async APIs differ")
print("  2. Try modifying app.py and see if the tests still pass")
print("  3. Try adding a new endpoint and writing a test for it")